# Šora algoritms

Šora algoritms tiek izmantots, lai faktorizētu skaitli $N$ (atrastu netriviālus dalītājus) polinomiālā laikā ar kvantu datora palīdzību. Klasiski liela $N$ faktorizācija ir smaga un laikietilpīga, bet Šora algoritms reducē problēmu uz kārtas atrašanu.

### Galvenā ideja aiz algoritma
Izvēlamies nejaušu $a$, kur $1<a<N$, un $gcd(a,N)=1$.  
Mērķis ir atrast mazāko $r$, kam
$$
a^r\equiv1\pmod N.
$$

Ja $r$ ir pāra un $a^{r/2}\not\equiv -1\pmod N,$ tad iegūstam netriviālus dalītājus:
$$
p=\gcd(a^{r/2}-1,N),\qquad
q=\gcd(a^{r/2}+1,N).
$$


### Algoritma kodols

Definējam unitāru operatoru
$$
U_a|y\rangle=|ay\bmod N\rangle.
$$

Šim operatoram ir eigenstāvokļi $|u_s\rangle$, $s=0,\dots,r-1$, ar
$$
U_a|u_s\rangle=e^{2\pi i s/r}|u_s\rangle.
$$

Tātad eigenfāzes ir
$$
\phi_s=\frac{s}{r}.
$$

Ar QPE varam novērtēt šīs fāzes un no tām iegūt informāciju par $r$.
### Eigenstāvokļu konstrukcija
Eigenstāvokļus var izteikt:
$$
|u_s\rangle=
\frac{1}{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}\,|a^k \bmod N\rangle,
\qquad s=0,\dots,r-1.
$$

Lai pārbaudītu vai tas patiešam ir eigenstāvoklis, pielietojam $U\equiv U_a$:
$$
U|u_s\rangle=
U\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}|a^k\bmod N\rangle
=
\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}U|a^k\bmod N\rangle.
$$

Tā kā $U|a^k\bmod N\rangle=|a^{k+1}\bmod N\rangle,$ iegūstam
$$
U|u_s\rangle=
\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i sk/r}|a^{k+1}\bmod N\rangle.
$$

Izceļam globālo fāzi, lietojot
$$
e^{-2\pi i sk/r}=e^{2\pi i s/r}\,e^{-2\pi i s(k+1)/r}.
$$
Tāpēc
$$
U|u_s\rangle
=
e^{2\pi i s/r}\,
\frac1{\sqrt r}\sum_{k=0}^{r-1}
e^{-2\pi i s(k+1)/r}|a^{k+1}\bmod N\rangle.
$$

Pārsaucam indeksu $k+1=j$ (mod $r$):
$$
U|u_s\rangle
=
e^{2\pi i s/r}\,
\frac1{\sqrt r}\sum_{j=0}^{r-1}
e^{-2\pi i sj/r}|a^j\bmod N\rangle
=
e^{2\pi i s/r}|u_s\rangle.
$$

Tātad:
$$
U|u_s\rangle=e^{2\pi i s/r}|u_s\rangle.
$$

### Fāzes aprēķins

Pieņemsim, ka fāzes reģistrā ir $t$ kubiti, palīgreģistrā sākumā  ir $|1\rangle$.

Sākuma stāvoklis:
$$
|\psi_0\rangle = |0\rangle^{\otimes t}\otimes |1\rangle.
$$

Pēc Hadamard vārtiem uz fāzes reģistra iegūstam superpozīciju:
$$
|\psi_1\rangle = \frac{1}{2^{t/2}}\sum_{x=0}^{2^t-1}|x\rangle\otimes |1\rangle.
$$

Izmantojam identitāti:
$$
|1\rangle = \frac{1}{\sqrt r}\sum_{s=0}^{r-1}|u_s\rangle,
$$
tad varam izteikt

$$
|\psi_1\rangle
=
\frac{1}{\sqrt r}\sum_{s=0}^{r-1}
\left(
\frac{1}{2^{t/2}}\sum_{x=0}^{2^t-1}|x\rangle
\right)\otimes |u_s\rangle.
$$

Pēc kontrolētajiem $U_a^{2^k}$:

$$
|x\rangle|u_s\rangle \mapsto e^{2\pi i x s/r}|x\rangle|u_s\rangle,
$$

un tātad

$$
|\psi_2\rangle
=
\frac{1}{\sqrt r}\sum_{s=0}^{r-1}
\left(
\frac{1}{2^{t/2}}\sum_{x=0}^{2^t-1}e^{2\pi i x s/r}|x\rangle
\right)\otimes |u_s\rangle.
$$

Pēc inversā QFT uz fāzes reģistra mērījums dod $m$, kur
$$
\frac{m}{2^t}\approx \frac{s}{r}.
$$

tad izmantojam continued fractions, lai atrastu racionālu tuvinājumu:
$$
\frac{s'}{r'},\qquad r'<N.
$$

Pēc tam kandidātu $r'$ pārbaudām ievietojot to $a^{r'}\equiv 1 \pmod N\;$

Ja pārbaude neizdodas, izmanto nākamo daļas tuvinājumu vai veic jaunu mērījumu.


In [28]:
import numpy as np
from math import gcd, ceil, log2
from fractions import Fraction

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate, UnitaryGate
from qiskit_aer import AerSimulator


N = 27
a = 4
t = 8                     # count (fāzes) reģistra kubiti
n = max(4, ceil(log2(N)))  # work (palīgkubiti) reģistra kubiti
shots = 8192



def make_Ua_gate(a, N, n):
    dim = 2 ** n #stāvokļu skaits palīgreģistrā
    U = np.zeros((dim, dim), dtype=complex) # izveido tukšu matricu

    for y in range(dim): 
        if y < N:
            y2 = (a * y) % N 
        else:
            y2 = y
        U[y2, y] = 1.0 #atzīmē pāreju 1

    return UnitaryGate(U, label=f"Ua({a},{N})")




def run_shor(N, a, t, n, shots=4096):
    if gcd(a, N) != 1:
        g = gcd(a, N) 
        print(f"gcd(a,N) = {g} -> faktori uzreiz: {g} un {N//g}")
        return
    if N > 2**n:
        raise ValueError(f"{N} neietilpst {n} kubitos.")

    Q = 2 ** t

    qc = QuantumCircuit(t + n, t)
    count = list(range(t)) #count kubitu indeksi
    work = list(range(t, t + n)) #work kubitu indeksi

    #work iestata uz |1>
    qc.x(work[0])

    #count superpozīcija
    for q in count:
        qc.h(q)

    #controlled-U^(2^k)
    Ua = make_Ua_gate(a, N, n)
    U_pow = Ua
    for k in range(t): #count kubitiem
        qc.append(U_pow.control(1), [count[k]] + work) #pievieno control kubitu
        U_pow = U_pow.power(2) 

    #inverse QFT uz count
    qc.append(QFTGate(t).inverse(), count)

    #mēram count
    qc.measure(count, range(t))




    
    sim = AerSimulator()
    tqc = transpile(qc, sim, optimization_level=1)
    print(tqc.num_qubits)
    result = sim.run(tqc, shots=shots).result()
    counts = result.get_counts()

    # Paņem visbiežāko mērījumu
    m_bits = max(counts, key=counts.get)
    m = int(m_bits, 2)
    
    # m/2^t
    x = m / Q
    
    # Aptuvens r no continued fractions
    r = Fraction(x).limit_denominator(N - 1).denominator
    
    print("m_bits =", m_bits, " : ", counts[m_bits])
    print("m =", m, "x =", x, "r =", r)
    print("a^r mod N =", pow(a, r, N))
    
    # Mēģinam faktorus
    if r % 2 == 0 and pow(a, r, N) == 1 and pow(a, r//2, N) != N - 1:
        p = gcd(pow(a, r//2) - 1, N)
        q = gcd(pow(a, r//2) + 1, N)
        print("Faktori:", p, q)
    else:
        print("Šoreiz nesanāca. Pamēģini vēlreiz (vai citu a / lielāku t).")
    
    

run_shor(N, a, t, n, shots)


13
m_bits = 11000111  :  915
m = 199 x = 0.77734375 r = 9
a^r mod N = 1
Šoreiz nesanāca. Pamēģini vēlreiz (vai citu a / lielāku t).
